##### Parameters

In [0]:
dbutils.widgets.text("env", "dev")
dbutils.widgets.text("file", "Legacy_Purchase_Orders_20260112.csv")

##### Configuration

In [0]:
spark.conf.set(
    "spark.sql.legacy.timeParserPolicy", "LEGACY"
)

#### Read Data

In [0]:
file = dbutils.widgets.get("file")
procurement_df = spark.read\
    .option("header", "true")\
    .option("inferSchema", "false")\
    .option("ignoreLeadingWhiteSpace", True)\
    .option("ignoreTrailingWhiteSpace", True)\
    .format("csv")\
    .load(f"s3://purchase-orders-aws/bronze/{file}")

#### Schema

In [0]:
from pyspark.sql.types import *

# schema as per requirement
schema = StructType([
    StructField('RECORD TYPE', StringType(), nullable=True),
    StructField('DOCUMENT NUMBER', IntegerType(), nullable=True),
    StructField('SOURCE DOCUMENT TYPE', StringType(), nullable=True),
    StructField('DOCUMENT DESCRIPTION', StringType(), nullable=True),
    StructField('UNIQUE ID', StringType(), nullable=False),
    StructField('REQUISITION NUMBER', IntegerType(), nullable=True),
    StructField('INPUT DATE', DateType(), nullable=True),
    StructField('TOTAL AMOUNT', DoubleType(), nullable=True),
    StructField('DEPARTMENT NUMBER', IntegerType(), nullable=True),
    StructField('DEPARTMENT NAME', StringType(), nullable=True),
    StructField('COST CENTER', IntegerType(), nullable=True),
    StructField('COST CENTER NAME', StringType(), nullable=True),
    StructField('INPUT BY', StringType(), nullable=True),
    StructField('PURCHASING AGENT', StringType(), nullable=True),
    StructField('DOCUMENT TYPE CODE', StringType(), nullable=True),
    StructField('DOCUMENT TYPE DESCRIPTION', StringType(), nullable=True),
    StructField('DOCUMENT STATUS CODE', IntegerType(), nullable=True),
    StructField('DOCUMENT STATUS DESCRIPTION', StringType(), nullable=True),
    StructField('PO CATEGORY CODE', StringType(), nullable=True),
    StructField('PO CATEGORY DESCRIPTION', StringType(), nullable=True),
    StructField('VOUCHED AMOUNT', DoubleType(), nullable=True),
    StructField('VENDOR NUMBER', IntegerType(), nullable=True),
    StructField('START DATE', DateType(), nullable=True),
    StructField('EXPIRATION DATE', DateType(), nullable=True),
    StructField('EXTENSION DATE', DateType(), nullable=True),
    StructField('ANNUAL CONTRACT', IntegerType(), nullable=True),
    StructField('VENDOR NAME 1', StringType(), nullable=True),
    StructField('VENDOR NAME 2', StringType(), nullable=True),
    StructField('VENDOR ADDRESS 1', StringType(), nullable=True),
    StructField('VENDOR ADDRESS 2', StringType(), nullable=True),
    StructField('VENDOR CITY', StringType(), nullable=True),
    StructField('VENDOR STATE', StringType(), nullable=True),
    StructField('VENDOR ZIP', StringType(), nullable=True),
    StructField('VENDOR CONTACT NAME', StringType(), nullable=True),
    StructField('VENDOR CONTACT TITLE', StringType(), nullable=True),
    StructField('VENDOR CONTACT PHONE', LongType(), nullable=True),
    StructField('VENDOR CONTACT EXTENSION', LongType(), nullable=True),
    StructField('VENDOR TYPE', StringType(), nullable=True),
    StructField('GENDER', StringType(), nullable=True),
    StructField('ETHNICITY', StringType(), nullable=True),
    StructField('STATUS', StringType(), nullable=True),
    StructField('CLASS', StringType(), nullable=True),
    StructField('GEOGRAPHIC AREA', StringType(), nullable=True),
    StructField('INDEPENDENT CONTRACTOR', BooleanType(), nullable=True),
    StructField('MINORITY', StringType(), nullable=True),
    StructField('VENDOR MINORITY DESCRIPTION', StringType(), nullable=True),
    StructField('DISADVANTAGED', BooleanType(), nullable=True),
    StructField('DISABLED VETERAN', BooleanType(), nullable=True),
    StructField('SB DISABLED VET', BooleanType(), nullable=True),
    StructField('SB MINORITY', BooleanType(), nullable=True),
    StructField('SB MINORITY WOMAN', BooleanType(), nullable=True),
    StructField('SB NON-MINORITY', BooleanType(), nullable=True),
    StructField('SB DISADVANTAGED', BooleanType(), nullable=True),
    StructField('SB VETERAN', BooleanType(), nullable=True),
    StructField('SB WOMAN', BooleanType(), nullable=True),
    StructField('TOTAL ITEMS', IntegerType(), nullable=True),
    StructField('PO BALANCE', DoubleType(), nullable=True),
    StructField('ITEM NUMBER', IntegerType(), nullable=True),
    StructField('ITEM DESCRIPTION', StringType(), nullable=True),
    StructField('ITEM QUANTITY ORDERED', DoubleType(), nullable=True),
    StructField('ITEM UNIT OF MEASURE', StringType(), nullable=True),
    StructField('ITEM UNIT OF MEASURE DESCRIPTION', StringType(), nullable=True),
    StructField('ITEM UNIT COST', DoubleType(), nullable=True),
    StructField('ITEM TOTAL COST', DoubleType(), nullable=True),
    StructField('COMMODITY CODE', IntegerType(), nullable=True),
    StructField('COMMODITY DESCRIPTION', StringType(), nullable=True),
    StructField('EXPENSE TYPE', IntegerType(), nullable=True),
    StructField('EXPENSE TYPE DESCRIPTION', StringType(), nullable=True)
])

##### Apply Transformation Logic

In [0]:
from pyspark.sql import functions as F

if dbutils.widgets.get("file") == "source.csv":
    procurement_df = procurement_df.withColumn(
        "INPUT DATE",
        F.coalesce(
            F.to_timestamp("INPUT DATE", "yyyy-MM-dd'T'HH:mm:SSS"),
            F.to_timestamp("INPUT DATE", "yyyy-MM-dd'T'HH:mm:ss")
        ).cast("date")
    ).withColumn(
        "START DATE",
        F.coalesce(
            F.to_timestamp("START DATE", "yyyy-MM-dd'T'HH:mm:SSS"),
            F.to_timestamp("START DATE", "yyyy-MM-dd'T'HH:mm:ss")
        ).cast("date")
    ).withColumn(
        "EXPIRATION DATE",
        F.coalesce(
            F.to_timestamp("EXPIRATION DATE", "yyyy-MM-dd'T'HH:mm:SSS"),
            F.to_timestamp("EXPIRATION DATE", "yyyy-MM-dd'T'HH:mm:ss")
        ).cast("date")
    ).withColumn(
        "EXTENSION DATE",
        F.coalesce(
            F.to_timestamp("EXTENSION DATE", "yyyy-MM-dd'T'HH:mm:SSS"),
            F.to_timestamp("EXTENSION DATE", "yyyy-MM-dd'T'HH:mm:ss")
        ).cast("date")
    )

else:
    procurement_df = procurement_df.withColumn(
        "INPUT DATE",
        F.coalesce(
            F.to_timestamp("INPUT DATE", "MM/dd/yyyy"),
            F.to_timestamp("INPUT DATE", "yyyy-MM-dd'T'HH:mm:ss"),
            F.to_timestamp("INPUT DATE", "yyyy-MM-dd'T'HH:mm:ss.SSS")
        ).cast("date")
    )

In [0]:
from pyspark.sql import functions as F
# remove intial letter from PURCHASE ORDER NUMBER most frequently P0 and put P in new column to adhere new schema format
# remove RQ0 from REQUISITION NUMBER and extract only numeric values
# remove $ sign and commas from TOTAL AMOUNT, VOUCHED AMOUNT, PO BALANCE, ITEM UNIT COST, ITEM TOTAL COST
# remove commas from VENDOR CONTACT PHONE, VENDOR CONTACT EXTENSION
# extract only numeric values from ITEM QUANTITY ORDERED
# change INDEPENDENT CONTRACTOR, DISADVANTAGED, DISABLED VETERAN, SB DISABLED VET, 
# SB MINORITY, SB MINORITY WOMAN, SB NON-MINORITY, SB DISADVANTAGED, SB VETERAN, SB WOMAN to bool

if "PURCHASE ORDER NUMBER" in procurement_df.columns:
    procurement_df = procurement_df\
                .withColumn('SOURCE DOCUMENT TYPE', F.substring(F.col("PURCHASE ORDER NUMBER"), 1, 1))\
                .withColumn('PURCHASE ORDER NUMBER', F.regexp_replace(F.col("PURCHASE ORDER NUMBER"), "^PO", ""))\
                .withColumn('REQUISITION NUMBER', F.regexp_replace(F.col('REQUISITION NUMBER'), r"[^0-9]", ""))\
                .withColumn('TOTAL AMOUNT', F.regexp_replace(F.regexp_replace(F.col('TOTAL AMOUNT'), r"\$", ""), ",", ""))\
                .withColumn('VOUCHED AMOUNT', F.regexp_replace(F.regexp_replace(F.col('VOUCHED AMOUNT'), r"\$", ""), ",", ""))\
                .withColumn('PO BALANCE', F.regexp_replace(F.regexp_replace(F.col('PO BALANCE'), r"\$", ""), ",", ""))\
                .withColumn('ITEM UNIT COST', F.regexp_replace(F.regexp_replace(F.col('ITEM UNIT COST'), r"\$", ""), ",", ""))\
                .withColumn('ITEM TOTAL COST', F.regexp_replace(F.regexp_replace(F.col('ITEM TOTAL COST'), r"\$", ""), ",", ""))\
                .withColumn('VENDOR CONTACT PHONE', F.regexp_replace(F.col('VENDOR CONTACT PHONE'), ",", ""))\
                .withColumn('VENDOR CONTACT EXTENSION', F.regexp_replace(F.col('VENDOR CONTACT EXTENSION'), ",", ""))\
                .withColumn('ITEM QUANTITY ORDERED', F.regexp_replace(F.col('ITEM QUANTITY ORDERED'), r"[^0-9.]", ""))
if "PURCHASE ORDER NUMBER" not in procurement_df.columns:
    procurement_df = procurement_df\
                        .withColumn('INDEPENDENT CONTRACTOR',
                            F.when(F.col('INDEPENDENT CONTRACTOR') == 'Y', True)\
                            .otherwise(False)
                        )\
                        .withColumn('DISADVANTAGED',
                            F.when(F.col('DISADVANTAGED') == 'Y', True)\
                            .otherwise(False)
                        )\
                        .withColumn('DISABLED VETERAN',
                            F.when(F.col('DISABLED VETERAN') == 'Y', True)\
                            .otherwise(False)
                        )\
                        .withColumn('SB DISABLED VET',
                            F.when(F.col('SB DISABLED VET') == 'Y', True)\
                            .otherwise(False)
                        )\
                        .withColumn('SB MINORITY',
                            F.when(F.col('SB MINORITY') == 'Y', True)\
                            .otherwise(False)
                        )\
                        .withColumn('SB MINORITY WOMAN',
                            F.when(F.col('SB MINORITY WOMAN') == 'Y', True)\
                            .otherwise(False)
                        )\
                        .withColumn('SB NON-MINORITY',
                            F.when(F.col('SB NON-MINORITY') == 'Y', True)\
                            .otherwise(False)
                        )\
                        .withColumn('SB DISADVANTAGED',
                            F.when(F.col('SB DISADVANTAGED') == 'Y', True)\
                            .otherwise(False)
                        )\
                        .withColumn('SB VETERAN',
                            F.when(F.col('SB VETERAN') == 'Y', True)\
                            .otherwise(False)
                        )\
                        .withColumn('SB WOMAN',
                            F.when(F.col('SB WOMAN') == 'Y', True)\
                            .otherwise(False)
                        )

##### Mapping

In [0]:
# Rename old columns with new column mappings to adhere common schema
procurement_df = procurement_df.withColumnRenamed('PURCHASE ORDER NUMBER', 'DOCUMENT NUMBER')\
                .withColumnRenamed('PO TYPE CODE', 'DOCUMENT TYPE CODE')\
                .withColumnRenamed('PO TYPE DESCRIPTION', 'DOCUMENT TYPE DESCRIPTION')\
                .withColumnRenamed('PO STATUS CODE', 'DOCUMENT STATUS CODE')\
                .withColumnRenamed('PO STATUS DESCRIPTION', 'DOCUMENT STATUS DESCRIPTION')\
                .withColumnRenamed('VENDOR MINORITY CODE', 'MINORITY')
# apply datatypes to existing columns in dataframe as per schema
# refer to docs/schemas/extraction-schema-mapping for understanding
# use try_cast to handle malformed values gracefully
try:
    for field in schema.fields:
        if field.name not in procurement_df.columns:
            procurement_df = procurement_df.withColumn(field.name, F.lit(None))
        else:
            procurement_df = procurement_df.withColumn(field.name, F.expr(f"try_cast(`{field.name}` as {field.dataType.simpleString()})"))
except Exception as e:
    print(e)

procurement_df = procurement_df.select([fi.name for fi in schema.fields])

##### Add Signature

In [0]:
from pyspark.sql.window import Window
# UNIQUE ID is a business key having highest cardinality which was deprecated in new schema 
# but due to high cardinality we are adding it using a business logic used earlier to generate UNIQUE ID
# Along with it we're adding current date in INGESTED_AT_ column
# We are also adding FILE_SOURCE with three value LEGACY_V1, NEW_V2 for source datasets 
# and API for API source to track data lineage
window = Window.partitionBy('DOCUMENT NUMBER', 'ITEM NUMBER').orderBy(F.col('INPUT DATE').desc())
if procurement_df.filter(F.col('UNIQUE ID').isNull()).count() > 0:
    procurement_df = procurement_df.withColumn(
        'UNIQUE ID',
        F.when(
            F.col('ITEM NUMBER') == 0, 
            F.concat(F.col('SOURCE DOCUMENT TYPE'), F.lit("O"), F.col('DOCUMENT NUMBER').cast("string"), F.row_number().over(window))
        ).otherwise(
            F.concat(F.col('SOURCE DOCUMENT TYPE'), F.lit("O"), F.col('DOCUMENT NUMBER').cast("string"), F.col('ITEM NUMBER').cast("string"), F.row_number().over(window))
        )
    )


is_input_by_empty = procurement_df.select('INPUT BY').na.drop().isEmpty()
is_sb_woman_empty = procurement_df.select('SB WOMAN').na.drop().isEmpty()

procurement_df = procurement_df.withColumn('_INGESTED_AT_', F.current_date()) \
    .withColumn('__FILE_SOURCE__', 
        F.when(F.lit(is_input_by_empty) & F.lit(dbutils.widgets.get("file") == "Purchase_Orders_and_Contracts_V2.csv"), 'NEW_V2')
         .when(F.lit(is_sb_woman_empty), 'LEGACY_V1')
         .otherwise('API_V3')
    )

#### Create Table

In [0]:
%sql
CREATE TABLE IF NOT EXISTS PurchaseOrders
USING DELTA
LOCATION "s3://purchase-orders-aws/silver"
AS
SELECT
    CAST(NULL AS DATE) AS INPUT_DATE,
    CAST(NULL AS STRING) AS RECORD_TYPE,
    CAST(NULL AS INT) AS DOCUMENT_NUMBER,
    CAST(NULL AS STRING) AS SOURCE_DOCUMENT_TYPE,
    CAST(NULL AS STRING) AS DOCUMENT_DESCRIPTION,
    CAST('1' AS STRING) AS UNIQUE_ID,
    CAST(NULL AS INT) AS REQUISITION_NUMBER,
    CAST(NULL AS DOUBLE) AS TOTAL_AMOUNT,
    CAST(NULL AS INT) AS DEPARTMENT_NUMBER,
    CAST(NULL AS STRING) AS DEPARTMENT_NAME,
    CAST(NULL AS INT) AS COST_CENTER,
    CAST(NULL AS STRING) AS COST_CENTER_NAME,
    CAST(NULL AS STRING) AS INPUT_BY,
    CAST(NULL AS STRING) AS PURCHASING_AGENT,
    CAST(NULL AS STRING) AS DOCUMENT_TYPE_CODE,
    CAST(NULL AS STRING) AS DOCUMENT_TYPE_DESCRIPTION,
    CAST(NULL AS INT) AS DOCUMENT_STATUS_CODE,
    CAST(NULL AS STRING) AS DOCUMENT_STATUS_DESCRIPTION,
    CAST(NULL AS STRING) AS PO_CATEGORY_CODE,
    CAST(NULL AS STRING) AS PO_CATEGORY_DESCRIPTION,
    CAST(NULL AS DOUBLE) AS VOUCHED_AMOUNT,
    CAST(NULL AS INT) AS VENDOR_NUMBER,
    CAST(NULL AS STRING) AS START_DATE,
    CAST(NULL AS STRING) AS EXPIRATION_DATE,
    CAST(NULL AS STRING) AS EXTENSION_DATE,
    CAST(NULL AS INT) AS ANNUAL_CONTRACT,
    CAST(NULL AS STRING) AS VENDOR_NAME_1,
    CAST(NULL AS STRING) AS VENDOR_NAME_2,
    CAST(NULL AS STRING) AS VENDOR_ADDRESS_1,
    CAST(NULL AS STRING) AS VENDOR_ADDRESS_2,
    CAST(NULL AS STRING) AS VENDOR_CITY,
    CAST(NULL AS STRING) AS VENDOR_STATE,
    CAST(NULL AS STRING) AS VENDOR_ZIP,
    CAST(NULL AS STRING) AS VENDOR_CONTACT_NAME,
    CAST(NULL AS STRING) AS VENDOR_CONTACT_TITLE,
    CAST(NULL AS BIGINT) AS VENDOR_CONTACT_PHONE,
    CAST(NULL AS BIGINT) AS VENDOR_CONTACT_EXTENSION,
    CAST(NULL AS STRING) AS VENDOR_TYPE,
    CAST(NULL AS STRING) AS GENDER,
    CAST(NULL AS STRING) AS ETHNICITY,
    CAST(NULL AS STRING) AS STATUS,
    CAST(NULL AS STRING) AS CLASS,
    CAST(NULL AS STRING) AS GEOGRAPHIC_AREA,
    CAST(NULL AS BOOLEAN) AS INDEPENDENT_CONTRACTOR,
    CAST(NULL AS STRING) AS MINORITY,
    CAST(NULL AS STRING) AS VENDOR_MINORITY_DESCRIPTION,
    CAST(NULL AS BOOLEAN) AS DISADVANTAGED,
    CAST(NULL AS BOOLEAN) AS DISABLED_VETERAN,
    CAST(NULL AS BOOLEAN) AS SB_DISABLED_VET,
    CAST(NULL AS BOOLEAN) AS SB_MINORITY,
    CAST(NULL AS BOOLEAN) AS SB_MINORITY_WOMAN,
    CAST(NULL AS BOOLEAN) AS SB_NON_MINORITY,
    CAST(NULL AS BOOLEAN) AS SB_DISADVANTAGED,
    CAST(NULL AS BOOLEAN) AS SB_VETERAN,
    CAST(NULL AS BOOLEAN) AS SB_WOMAN,
    CAST(NULL AS INT) AS TOTAL_ITEMS,
    CAST(NULL AS DOUBLE) AS PO_BALANCE,
    CAST(NULL AS INT) AS ITEM_NUMBER,
    CAST(NULL AS STRING) AS ITEM_DESCRIPTION,
    CAST(NULL AS INT) AS ITEM_QUANTITY_ORDERED,
    CAST(NULL AS STRING) AS ITEM_UNIT_OF_MEASURE,
    CAST(NULL AS STRING) AS ITEM_UNIT_OF_MEASURE_DESCRIPTION,
    CAST(NULL AS DOUBLE) AS ITEM_UNIT_COST,
    CAST(NULL AS DOUBLE) AS ITEM_TOTAL_COST,
    CAST(NULL AS INT) AS COMMODITY_CODE,
    CAST(NULL AS STRING) AS COMMODITY_DESCRIPTION,
    CAST(NULL AS INT) AS EXPENSE_TYPE,
    CAST(NULL AS STRING) AS EXPENSE_TYPE_DESCRIPTION,
    CAST(NULL AS TIMESTAMP) AS _INGESTED_AT_,
    CAST(NULL AS STRING) AS __FILE_SOURCE__
WHERE 1 = 0

#### Write

In [0]:
from delta.tables import DeltaTable
purchase_orders = DeltaTable.forName(spark, 'PurchaseOrders')

purchase_orders.alias('sink').merge(
    source = procurement_df.alias('source'),
    condition = "sink.`UNIQUE_ID` = source.`UNIQUE ID`"
)\
.whenMatchedUpdate(set = {

}) \
.whenNotMatchedInsert(values={
    "RECORD_TYPE": F.col("RECORD TYPE"),
    "DOCUMENT_NUMBER": F.col("DOCUMENT NUMBER"),
    "REQUISITION_NUMBER": F.col("REQUISITION NUMBER"),
    "INPUT_DATE": F.col("INPUT DATE"),
    "TOTAL_AMOUNT": F.col("TOTAL AMOUNT"),
    "DEPARTMENT_NUMBER": F.col("DEPARTMENT NUMBER"),
    "DEPARTMENT_NAME": F.col("DEPARTMENT NAME"),
    "COST_CENTER": F.col("COST CENTER"),
    "COST_CENTER_NAME": F.col("COST CENTER NAME"),
    "INPUT_BY": F.col("INPUT BY"),
    "PURCHASING_AGENT": F.col("PURCHASING AGENT"),
    "DOCUMENT_TYPE_CODE": F.col("DOCUMENT TYPE CODE"),
    "DOCUMENT_TYPE_DESCRIPTION": F.col("DOCUMENT TYPE DESCRIPTION"),
    "PO_CATEGORY_CODE": F.col("PO CATEGORY CODE"),
    "PO_CATEGORY_DESCRIPTION": F.col("PO CATEGORY DESCRIPTION"),
    "DOCUMENT_STATUS_CODE": F.col("DOCUMENT STATUS CODE"),
    "DOCUMENT_STATUS_DESCRIPTION": F.col("DOCUMENT STATUS DESCRIPTION"),
    "VOUCHED_AMOUNT": F.col("VOUCHED AMOUNT"),
    "VENDOR_NUMBER": F.col("VENDOR NUMBER"),
    "VENDOR_NAME_1": F.col("VENDOR NAME 1"),
    "VENDOR_NAME_2": F.col("VENDOR NAME 2"),
    "VENDOR_ADDRESS_1": F.col("VENDOR ADDRESS 1"),
    "VENDOR_ADDRESS_2": F.col("VENDOR ADDRESS 2"),
    "VENDOR_CITY": F.col("VENDOR CITY"),
    "VENDOR_STATE": F.col("VENDOR STATE"),
    "VENDOR_ZIP": F.col("VENDOR ZIP"),
    "VENDOR_CONTACT_NAME": F.col("VENDOR CONTACT NAME"),
    "VENDOR_CONTACT_TITLE": F.col("VENDOR CONTACT TITLE"),
    "VENDOR_CONTACT_PHONE": F.col("VENDOR CONTACT PHONE"),
    "VENDOR_CONTACT_EXTENSION": F.col("VENDOR CONTACT EXTENSION"),
    "MINORITY": F.col("MINORITY"),
    "VENDOR_MINORITY_DESCRIPTION": F.col("VENDOR MINORITY DESCRIPTION"),
    "TOTAL_ITEMS": F.col("TOTAL ITEMS"),
    "PO_BALANCE": F.col("PO BALANCE"),
    "ITEM_NUMBER": F.col("ITEM NUMBER"),
    "ITEM_DESCRIPTION": F.col("ITEM DESCRIPTION"),
    "ITEM_QUANTITY_ORDERED": F.round(F.col("ITEM QUANTITY ORDERED"), 0).cast('int'),
    "ITEM_UNIT_OF_MEASURE": F.col("ITEM UNIT OF MEASURE"),
    "ITEM_UNIT_OF_MEASURE_DESCRIPTION": F.col("ITEM UNIT OF MEASURE DESCRIPTION"),
    "ITEM_UNIT_COST": F.col("ITEM UNIT COST"),
    "ITEM_TOTAL_COST": F.col("ITEM TOTAL COST"),
    "UNIQUE_ID": F.col("UNIQUE ID"),
    "SOURCE_DOCUMENT_TYPE": F.col("SOURCE DOCUMENT TYPE"),
    "COMMODITY_CODE": F.col("COMMODITY CODE"),
    "COMMODITY_DESCRIPTION": F.col("COMMODITY DESCRIPTION"),
    "EXPENSE_TYPE": F.col("EXPENSE TYPE"),
    "EXPENSE_TYPE_DESCRIPTION": F.col("EXPENSE TYPE DESCRIPTION"),
    "START_DATE": F.col("START DATE").cast('string'),
    "EXPIRATION_DATE": F.col("EXPIRATION DATE").cast('string'),
    "EXTENSION_DATE": F.col("EXTENSION DATE").cast('string'),
    "ANNUAL_CONTRACT": F.col("ANNUAL CONTRACT"),
    "VENDOR_TYPE": F.col("VENDOR TYPE"),
    "GENDER": F.col("GENDER"),
    "ETHNICITY": F.col("ETHNICITY"),
    "STATUS": F.col("STATUS"),
    "CLASS": F.col("CLASS"),
    "GEOGRAPHIC_AREA": F.col("GEOGRAPHIC AREA"),
    "INDEPENDENT_CONTRACTOR": F.col("INDEPENDENT CONTRACTOR"),
    "DISADVANTAGED": F.col("DISADVANTAGED"),
    "DISABLED_VETERAN": F.col("DISABLED VETERAN"),
    "SB_DISABLED_VET": F.col("SB DISABLED VET"),
    "SB_MINORITY": F.col("SB MINORITY"),
    "SB_MINORITY_WOMAN": F.col("SB MINORITY WOMAN"),
    "SB_NON_MINORITY": F.col("SB NON-MINORITY"),
    "SB_DISADVANTAGED": F.col("SB DISADVANTAGED"),
    "SB_VETERAN": F.col("SB VETERAN"),
    "SB_WOMAN": F.col("SB WOMAN"),
    "DOCUMENT_DESCRIPTION": F.col("DOCUMENT DESCRIPTION"),
    "_INGESTED_AT_": F.col("_INGESTED_AT_"),
    "__FILE_SOURCE__": F.col("__FILE_SOURCE__")
}) \
.execute()